# Project 01: Stock Screener: Rule-Based Screening & Rolling Evaluation
**Objective:** Expand the asset universe and implement a rule-based filtering logic based on volatility, momentum, and drawdown metrics, evaluated on a rolling basis.

In [1]:
import numpy as np
import pandas as pd
import duckdb
import yfinance as yf

In [2]:
# Extraction and Normalization of Data
tickers = ["NVDA", "MSFT", "MSTR", "JPM", "BAC", "MS", "LLY", "JNJ", "ABBV", "XOM", "CVX", "COP", "PG", "KO", "PEP", "SPY"]
start_date = "2022-06-01"
end_date = "2025-01-01"

market_data_wide = yf.download(tickers, start_date, end_date, auto_adjust=True, progress=False, multi_level_index=True)
market_data = market_data_wide.stack().reset_index()
market_data.columns = market_data.columns.str.lower()
market_data.columns.name = None
market_data = market_data[["date", "ticker", "open", "high", "low", "close", "volume"]]

# Generation of Analytical Base Table in-memory
query = """
WITH previous_prices AS (
    SELECT 
        date, ticker, open, high, low, close, volume,
        LAG(close) OVER (PARTITION BY ticker ORDER BY date) AS previous_close
    FROM market_data
)
SELECT 
    date, ticker, open, high, low, close, volume,
    (close / previous_close) - 1 AS daily_return
FROM previous_prices
ORDER BY ticker, date;
"""

daily_returns = duckdb.sql(query).df()

# Validation
assert daily_returns.shape[1] == 8, "Expected 8 columns in the Analytical Base Table."
assert daily_returns['daily_return'].isna().sum() == 16, "Expected exactly 16 NaNs for daily_return at t_0."
assert daily_returns.duplicated(subset=['date', 'ticker']).sum() == 0, "Duplicate composite primary keys found."

### Universe Expansion Strategy
The universe has been expanded from 3 to 16 highly liquid assets spanning 5 major GICS sectors (Tech, Financials, Healthcare, Energy, Consumer Staples), plus the SPY benchmark. 
*Rationale:* A 16-asset universe provides sufficient cross-sectional breadth to evaluate the screening logic dynamically, avoiding trivial pass/fail outcomes, while maintaining high liquidity constraints to simulate a realistic investable universe.

*Note on Data Ingestion:* The time window starts in June 2022 to account for the 126-day rolling window "burn-in" period, ensuring valid feature generation starting from early 2023.

In [3]:
screening_features = duckdb.sql("""
WITH lag_values AS (
    SELECT 
        date, ticker, close,

        LAG(close, 125) OVER (PARTITION BY ticker ORDER BY date) AS close_126d_ago,
        MAX(close) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN 125 PRECEDING AND CURRENT ROW) AS peak_126d,
        STDDEV_SAMP(daily_return) OVER(PARTITION BY ticker ORDER BY date ROWS BETWEEN 62 PRECEDING AND CURRENT ROW) AS raw_vol_63d,

        COUNT(close) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN 125 PRECEDING AND CURRENT ROW) AS count_126d,
        COUNT(daily_return) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN 62 PRECEDING AND CURRENT ROW) AS count_63d,

    FROM daily_returns
)
SELECT date, ticker,
CASE WHEN count_126d = 126 THEN (close / close_126d_ago) - 1 ELSE NULL END AS momentum_126d,
CASE WHEN count_63d = 63 THEN raw_vol_63d * SQRT(252) ELSE NULL END AS rolling_vol_63d,
CASE WHEN count_126d = 126 THEN (close / peak_126d) - 1 ELSE NULL END AS current_dd_126d
FROM lag_values
ORDER BY ticker, date
""").df()

# NOTE: arbitrary numbers (16, 62, 125) are hardcoded to the current universe size / window lengths. See README "Known Limitations".
assert screening_features['rolling_vol_63d'].isna().groupby(screening_features['ticker']).sum().min() >= 62, "Cold start for volatility not handled correctly."
assert screening_features['momentum_126d'].isna().groupby(screening_features['ticker']).sum().min() >= 125, "Cold start for momentum not handled correctly."
assert screening_features['current_dd_126d'].isna().groupby(screening_features['ticker']).sum().min() >= 125, "Cold start for MDD not handled correctly."
assert (screening_features['current_dd_126d'].dropna() <= 0).all(), "Max Drawdown cannot be positive."

# Screening Features
Three important features for the Stock Screener have been computed:

**1.** momentum_126d: calculate the return of a specific asset on a window of approximately six months (126 business days)

**2.** rolling_vol_63d: calculate the annualized volatility on a window of approximately three months (63 business days)

**3.** current_dd_126d: calculate the drawdown of the current value of the asset with respect to the maximum registered on a window of approximately six months

In [4]:
screener_results = duckdb.sql("""
WITH cross_sectional_features AS (
SELECT *,
QUANTILE_CONT(rolling_vol_63d, 0.5 ORDER BY rolling_vol_63d) OVER (PARTITION BY date) AS median
FROM screening_features
ORDER BY date, ticker
)
SELECT *,
momentum_126d > 0 AS pass_momentum,
rolling_vol_63d < median AS pass_volatility,
current_dd_126d >= -0.15 AS pass_drawdown,
COALESCE (momentum_126d > 0 AND rolling_vol_63d < median AND current_dd_126d >= -0.15, FALSE) AS is_investable
FROM cross_sectional_features
ORDER BY date
""").df()

# Total protection from cold-start: no NULL values may be investable
assert screener_results.loc[screener_results['momentum_126d'].isna(), 'is_investable'].sum() == 0, "Leakage: Assets in burn-in period passed the screener."

# Formal check of aggregate boolean logic
expected_investable = (
    screener_results['pass_momentum'] & 
    screener_results['pass_drawdown'] & 
    screener_results['pass_volatility']
).fillna(False)
assert (screener_results['is_investable'] == expected_investable).all(), "Boolean aggregation logic failed. Check NULL handling."

# Cross-sectional check: is_investable must actively filter (not all False, not all True)
print(f"Total historical data points: {len(screener_results)}")
print(f"Total investable signals generated: {screener_results['is_investable'].sum()}")
# Show the most recent positive signals
display(screener_results[screener_results['is_investable']].tail(10))

Total historical data points: 10400
Total investable signals generated: 3015


,date,ticker,momentum_126d,rolling_vol_63d,current_dd_126d,median,pass_momentum,pass_volatility,pass_drawdown,is_investable
10350,2024-12-26,SPY,0.112077,0.122431,-0.007315,0.233023,True,True,True,True
10356,2024-12-27,JNJ,0.006034,0.132259,-0.126456,0.234288,True,True,True,True
10358,2024-12-27,KO,0.001186,0.139630,-0.132242,0.234288,True,True,True,True
10365,2024-12-27,PG,0.054280,0.148571,-0.056594,0.234288,True,True,True,True
10366,2024-12-27,SPY,0.098111,0.124458,-0.017764,0.234288,True,True,True,True
10381,2024-12-30,PG,0.031625,0.151213,-0.070172,0.235432,True,True,True,True
10382,2024-12-30,SPY,0.078323,0.126607,-0.028973,0.235432,True,True,True,True
10388,2024-12-31,JNJ,0.008215,0.135014,-0.129045,0.232611,True,True,True,True
10397,2024-12-31,PG,0.035524,0.151398,-0.067056,0.232611,True,True,True,True
10398,2024-12-31,SPY,0.069627,0.125437,-0.032506,0.232611,True,True,True,True


In [ ]:
# Extract the End-of-Month (EoM) snapshots and aggregate the investable universe
monthly_portfolio = (
    screener_results
    .groupby(['ticker', pd.Grouper(key='date', freq='ME')]).last() # Isolate the last trading day per month
    .reset_index()                                                 # Flatten the MultiIndex
    .query("is_investable")                                        # Filter only positive signals (is_investable == True)
    .groupby('date')                                               # Group transversally by month
    .agg(
        portfolio_size=('ticker', 'count'),                        # Named Aggregation: Count
        investable_tickers=('ticker', list)                        # Named Aggregation: List compilation
    )
)

display(monthly_portfolio)

# Validation
assert monthly_portfolio.index.is_unique, "Error: Duplicate months found in the portfolio timeline."
assert monthly_portfolio.index.is_month_end.all(), "Error: Dates are not properly aligned to End-of-Month."
assert (monthly_portfolio['portfolio_size'] == monthly_portfolio['investable_tickers'].apply(len)).all(), "Error: Size metric mismatch."

,portfolio_size,investable_tickers
date,,
2022-11-30,7,"[ABBV, JNJ, JPM, KO, LLY, PEP, PG]"
2022-12-31,8,"[ABBV, JNJ, JPM, KO, LLY, PEP, PG, SPY]"
2023-01-31,5,"[ABBV, JPM, LLY, PG, SPY]"
2023-02-28,3,"[ABBV, JPM, PEP]"
2023-03-31,7,"[ABBV, CVX, KO, LLY, PEP, PG, SPY]"
2023-04-30,6,"[ABBV, KO, LLY, PEP, PG, SPY]"
2023-05-31,3,"[LLY, PEP, SPY]"
2023-06-30,5,"[LLY, MS, PEP, PG, SPY]"
2023-07-31,7,"[ABBV, JNJ, JPM, KO, PEP, PG, SPY]"


# Rolling Screening Evaluation & Periodic Rebalancing
**Objective:** Evaluate the stability and turnover of the screening rules over time by implementing a Discrete Rebalancing simulation (End-of-Month).

### Architectural Choices
* **Multi-Index Flattening:** Transitioned from SQL/DuckDB back to Pandas for downstream list-aggregation, utilizing `.reset_index()` and Named Aggregation (`.agg`).
* **Turnover Mitigation:** The screener is evaluated exclusively at the end of each calendar month (`pd.Grouper(freq='ME')`), avoiding the frictional costs (commissions, bid-ask spread, noise) associated with daily signal evaluation.
* **Regime Stability:** The output provides a chronological audit trail of the portfolio's composition, proving the dynamic scaling of risk exposure (e.g., dynamically reducing the investable universe during systemic market pullbacks).